# AQUAVIEW Python SDK — verification notebook

Exercises every part of the `aquaview` SDK end-to-end. Run it top to bottom.

- **Section 1 (public)** needs no credentials — it runs against the live catalog.
- **Section 2 (authenticated)** needs an API key: create one in the AQUAVIEW
  portal (Settings → API keys, with the **download** permission) and set
  `AQUAVIEW_API_KEY` before running.
- **Section 3 (async export)** additionally needs the jobs API change deployed.

A cell "passes" if it runs without raising. Errors surface as
`aquaview.AquaviewAPIError` with a `.status` / `.code` / `.message`.

In [ ]:
import sys, aquaview
print(sys.executable)
print(aquaview.__file__)


In [ ]:
# If running OUTSIDE this repo's virtualenv (e.g. Colab), install first —
# but note the new features need v0.5.0+. In the repo venv, SKIP this: it
# already has the dev version and pip would downgrade it.
# %pip install -q aquaview
import aquaview
print('SDK version:', aquaview.__version__)

## 1. Public surface — no API key


In [ ]:
client = aquaview.Client()

# STAC catalog search (delegates to pystac-client)
item = next(iter(client.search(limit=1, max_items=1).items()))
print('search        ->', item.collection_id, item.id)

In [ ]:
# Data sources registry + one source's detail
sources = client.get_sources()
print('get_sources   ->', len(sources), 'sources')
detail = client.get_source(sources[0]['source_id'])
print('get_source    ->', detail['source_id'], '| variables:', detail['canonical_variables'][:5])

In [ ]:
# Curated, theme-based collections
cols = client.get_curated_collections()
print('curated cols  ->', len(cols), '| e.g.', cols[0]['id'])

In [ ]:
# Recommendations: datasets similar to the search hit above
similar = client.get_similar(item.collection_id, item.id, limit=3)
print('get_similar   ->', len(similar), 'similar datasets')

In [ ]:
# Pull a tiny real data slice (limit=1) from a public source, as CSV bytes
src = sources[0]
csv_bytes = client.get_data(src['source_id'], [src['canonical_variables'][7]], limit=1, format='csv')
print('get_data      ->', len(csv_bytes), 'bytes')
print(csv_bytes.decode(errors='replace')[:200])

`get_schema` depends on the Beacon engine; it may return a 500 if Beacon is down. The SDK surfaces that as `AquaviewAPIError` — expected, not an SDK fault.


In [ ]:
try:
    print('get_schema    ->', list(client.get_schema()))
except aquaview.AquaviewAPIError as e:
    print('get_schema    -> API error (Beacon):', e.status, e.message)

## 2. Authenticated — set `AQUAVIEW_API_KEY`

These need a real API key. By default they hit the SDK's configured host; set `AQUAVIEW_API_URL` to target a different environment (e.g. staging while the jobs API change rolls out).

In [ ]:
# Set AQUAVIEW_API_KEY in your shell before launching (never hard-code it here):
#   export AQUAVIEW_API_KEY=...


In [ ]:
import os
assert os.environ.get('AQUAVIEW_API_KEY'), 'Set AQUAVIEW_API_KEY first'
# Optional: point at a non-default environment (host serves both /api and /stac)
base = os.environ.get('AQUAVIEW_API_URL')
kc = aquaview.Client(api_url=base, catalog_url=base + '/stac') if base else aquaview.Client()
print('get_usage     ->', kc.get_usage())

In [ ]:
# Natural-language query -> structured filters you can feed to search()
print('interpret     ->', kc.interpret('warm water off Florida in 2024'))

In [ ]:
# Chat: final answer, then the live streaming form
print('chat          ->', kc.chat('what glider data is off the US east coast?'))
for evt in kc.chat_stream('summarize WOD coverage in the Gulf of Mexico'):
    print('  ', evt['event'], str(evt['data'])[:100])

## 3. Async export — needs the jobs API change deployed

For pulls too big to stream inline. Requires an API key **and** the `feature/jobs-accept-api-keys` API change live with `JOBS_ENABLED=true`.


In [ ]:
job = kc.submit_export('gadr', ['temperature'])   # large, unbounded -> background job
print('submitted     ->', job.job_id)
job.wait()                                          # blocks until done (or raises JobError)
paths = job.download('export_out/')                 # one file per time shard
print('downloaded    ->', paths)

### What a passing run looks like

Section 1 prints results for every call (or a clean Beacon note for `get_schema`). Section 2 prints your usage, interpreted filters, and a chat answer. Section 3 prints a job id, then the downloaded part files. Any failure raises `aquaview.AquaviewAPIError` with the reason.
